In [4]:
# !pip install matplotlib numpy
# !pip install nltk

In [5]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords

In [6]:
df = pd.read_csv('D:\Professional\Learning at office\datas\Chat-Transcript_4d0c416d-ef0e-4fcd-b848-d80dcfda3f1e-1714557600000.csv')

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\geswaramoorthy\AppData\Local\Temp\ipykernel_8072\4145501746.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv('D:\Professional\Learning at office\datas\Chat-Transcript_4d0c416d-ef0e-4fcd-b848-d80dcfda3f1e-1714557600000.csv')
C:\Users\geswaramoorthy\AppData\Local\Temp\ipykernel_8072\4145501746.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('D:\Professional\Learning at office\datas\Chat-Transcript_4d0c416d-ef0e-4fcd-b848-d80dcfda3f1e-1714557600000.csv')


In [7]:
df.columns 

Index(['conversation_id', 'conversation_url', 'interaction_raw_id',
       'message_id', 'message_type', 'message_parts', 'created_time',
       'message_source', 'actor_id', 'actor_type', 'actor_sub_entity',
       'actor_email', 'actor_phone', 'actor_first_name', 'actor_last_name',
       'reference_id', 'channel_id', 'channel_name', 'detailed_message_type',
       'customer_id'],
      dtype='object')

In [8]:
# Download stopwords once if not done
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\geswaramoorthy\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [18]:
# Define custom stopword list
default_stopwords = set(stopwords.words('english'))
keep_words = {"who", "what", "how", "much", "where", "when", "why"}
custom_stopwords = default_stopwords - keep_words


# Filter only user messages
df_user = df[df['actor_type'] == 'user'].copy()

# Parse message_parts into structured columns
def parse_message_parts(message):
    try:
        items = ast.literal_eval(message)
    except Exception:
        return pd.Series([None] * 4, index=["text_content", "file_name", "file_url", "image_url"])

    text_content, file_name, file_url, image_url = [], [], [], []

    for item in items:
        if "text" in item:
            content = item["text"].get("content", "")
            if content:
                text_content.append(content)

        if "file" in item:
            file_name.append(item["file"].get("name", ""))
            file_url.append(item["file"].get("url", ""))

        if "image" in item:
            image_url.append(item["image"].get("url", ""))

    return pd.Series([
        ' '.join(text_content).strip() if text_content else None,
        '; '.join(file_name).strip() if file_name else None,
        '; '.join(file_url).strip() if file_url else None,
        '; '.join(image_url).strip() if image_url else None
    ], index=["text_content", "file_name", "file_url", "image_url"])

# Clean text_content
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    filtered = [word for word in words if word not in custom_stopwords]
    return ' '.join(filtered)

# Parse message_parts into new columns
parsed_cols = df_user['message_parts'].apply(parse_message_parts)
df_user = pd.concat([df_user.reset_index(drop=True), parsed_cols], axis=1)

# Clean text
df_user['cleaned_text'] = df_user['text_content'].apply(clean_text)


In [64]:
# grouped_df = df_user.groupby("conversation_id").agg({
#     'customer_id': 'first',
#     'conversation_url': 'first',
#     'channel_name': 'first',
#     'created_time': 'min',
#     'text_content': lambda x: ' '.join(str(i) for i in x if pd.notna(i)),
#     'cleaned_text': lambda x: ' '.join(str(i) for i in x if pd.notna(i)),
#     'file_name': lambda x: '; '.join(str(i) for i in x if pd.notna(i)),
#     'file_url': lambda x: '; '.join(str(i) for i in x if pd.notna(i)),
#     'image_url': lambda x: '; '.join(str(i) for i in x if pd.notna(i))
# }).reset_index()

grouped_df = df_user.groupby("conversation_id").agg({
    'customer_id': 'first',
    'conversation_url': 'first',
    'channel_name': 'first',
    'created_time': ['min', 'max'],
    'text_content': lambda x: ' '.join(str(i) for i in x if pd.notna(i)),
    'cleaned_text': lambda x: ' '.join(str(i) for i in x if pd.notna(i)),
    'file_name': lambda x: '; '.join(str(i) for i in x if pd.notna(i)),
    'file_url': lambda x: '; '.join(str(i) for i in x if pd.notna(i)),
    'image_url': lambda x: '; '.join(str(i) for i in x if pd.notna(i))
}).reset_index()

In [65]:
grouped_df.columns = [
    col if isinstance(col, str) else col[0] if col[1] == '' else f"{col[0]}"
    for col in grouped_df.columns
]
grouped_df = grouped_df.reset_index()

In [66]:
grouped_df.columns

Index(['index', 'conversation_id', 'customer_id', 'conversation_url',
       'channel_name', 'created_time', 'created_time', 'text_content',
       'cleaned_text', 'file_name', 'file_url', 'image_url'],
      dtype='object')

In [67]:
grouped_df.to_csv('./output/output_csv.csv')

In [57]:
grouped_df.head()

,index,conversation_id,customer_id_first,conversation_url_first,channel_name_first,created_time_min,created_time_max,text_content_<lambda>,cleaned_text_<lambda>,file_name_<lambda>,file_url_<lambda>,image_url_<lambda>
0,0,0027b761-a1a1-4b8e-b969-2c662bed4896,1dd6277a-b6a9-4845-8ac6-866f8d9298a9,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-09-14T12:24:22.453Z,2024-09-14T13:24:03.140Z,https://maps.google.com/maps?q=21.484920501709...,sara alabdouli august,,,
1,1,002c1c2d-4ccb-438d-a3eb-1ee95d028d4a,b075f1ee-e6f4-4bf7-9105-67dda7a37717,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-03T11:49:58.689Z,2024-05-03T13:48:41.612Z,https://maps.google.com/maps?q=25.074586868286...,,,,
2,2,004273d2-3d30-45ea-85a1-37e0d1769f49,2db48932-b623-4a70-9817-2ad80f8380b6,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-09T08:51:18.853Z,2024-05-10T06:05:55.867Z,May I know ur location At mussafah,may know ur location mussafah,,,
3,3,00482e27-7fe2-4095-a7dd-5ea9bf1f1751,63bf89a5-e185-40ed-9314-cb1116beddb3,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-31T09:05:39.050Z,2024-05-31T09:53:06.896Z,معليش معد ابي اتعلاج عندكم انا مستعجلة شوي ؟ T...,thank estradiol,,,
4,4,004b79a2-0426-449e-b2a7-647b90012a19,ad135963-e3c3-432c-829a-7328d2aa10c4,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-06T14:30:09.941Z,2024-05-06T15:12:11.295Z,Is there any additional cost to what I have pa...,additional cost what paid already,,,


In [ ]:
tags = ["product_inquiry","order_status","return_n_refund","Discount_n_promotion","purchase","Appointment_booking","service_availability","pricing_info","tech_support","billing_n_payments","Communication"]

### Topic Modeling with LDA (Latent Dirichlet Allocation)

In [32]:
!pip install scikit-learn

^C


  Using cached scikit_learn-1.6.1-cp313-cp313-win_amd64.whl.metadata (15 kB)
Using cached scikit_learn-1.6.1-cp313-cp313-win_amd64.whl (11.1 MB)


In [34]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

In [53]:
grouped_df.head()

,index,conversation_id,customer_id_first,conversation_url_first,channel_name_first,created_time_min,created_time_max,text_content_<lambda>,cleaned_text_<lambda>,file_name_<lambda>,file_url_<lambda>,image_url_<lambda>
0,0,0027b761-a1a1-4b8e-b969-2c662bed4896,1dd6277a-b6a9-4845-8ac6-866f8d9298a9,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-09-14T12:24:22.453Z,2024-09-14T13:24:03.140Z,https://maps.google.com/maps?q=21.484920501709...,sara alabdouli august,,,
1,1,002c1c2d-4ccb-438d-a3eb-1ee95d028d4a,b075f1ee-e6f4-4bf7-9105-67dda7a37717,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-03T11:49:58.689Z,2024-05-03T13:48:41.612Z,https://maps.google.com/maps?q=25.074586868286...,,,,
2,2,004273d2-3d30-45ea-85a1-37e0d1769f49,2db48932-b623-4a70-9817-2ad80f8380b6,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-09T08:51:18.853Z,2024-05-10T06:05:55.867Z,May I know ur location At mussafah,may know ur location mussafah,,,
3,3,00482e27-7fe2-4095-a7dd-5ea9bf1f1751,63bf89a5-e185-40ed-9314-cb1116beddb3,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-31T09:05:39.050Z,2024-05-31T09:53:06.896Z,معليش معد ابي اتعلاج عندكم انا مستعجلة شوي ؟ T...,thank estradiol,,,
4,4,004b79a2-0426-449e-b2a7-647b90012a19,ad135963-e3c3-432c-829a-7328d2aa10c4,https://feelvaleo-371220369404109218.freshchat...,WHATSAPP_+971549965988,2024-05-06T14:30:09.941Z,2024-05-06T15:12:11.295Z,Is there any additional cost to what I have pa...,additional cost what paid already,,,


In [68]:
texts = grouped_df['cleaned_text'].fillna("").tolist()

In [75]:
# Step 1: Vectorize text
vectorizer = CountVectorizer(
    max_df=0.9,
    min_df=2,
    stop_words='english'  # Built-in stopwords
)
doc_term_matrix = vectorizer.fit_transform(texts)

In [76]:
# Step 2: Fit LDA
lda_model = LatentDirichletAllocation(
    n_components=6,       # number of topics
    random_state=42,
    learning_method='batch'
)
lda_model.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=6, random_state=42)

In [77]:
# Step 3: Assign topics
topic_distributions = lda_model.transform(doc_term_matrix)
grouped_df['lda_topic'] = topic_distributions.argmax(axis=1)

In [78]:
# Step 4: Get top words for each topic
def get_topic_keywords(model, vectorizer, top_n=10):
    keywords = []
    for topic_idx, topic in enumerate(model.components_):
        top_features = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-top_n:]]
        keywords.append(", ".join(top_features))
    return keywords

In [79]:
topic_keywords = get_topic_keywords(lda_model, vectorizer)
grouped_df['lda_topic_keywords'] = grouped_df['lda_topic'].apply(lambda x: topic_keywords[x])

In [87]:
!pip install pyLDAvis

^C


In [88]:
import pyLDAvis
import pyLDAvis.sklearn

# Prepare the visualization
pyLDAvis.enable_notebook()  # If you're in a Jupyter environment
panel = pyLDAvis.sklearn.prepare(lda_model, doc_term_matrix, vectorizer)

# To show in Jupyter
panel 

ModuleNotFoundError: No module named 'pyLDAvis'